# 순환 깊이 스케일링 사다리 — Phase 1 (dense)

런타임: **TPU v6e-1**. 순서대로 실행한다.

1. 환경 + 테스트
2. 데이터 준비 (TinyStories = 게이트용, FineWeb-Edu = 사다리용)
3. **CU 소모율 10분 측정** — 여기 숫자를 보고 150m을 돌릴지 정한다
4. 15m 게이트 (TinyStories, recovery 0.41±0.04)
5. 통과했을 때만 사다리 (FineWeb-Edu)

세션이 끊기면 같은 셀을 다시 실행하면 된다. 전부 `--resume`이 붙어 있다.

In [ ]:
!pip -q install 'flax>=0.10' 'optax>=0.2.4' 'datasets>=3.0' 'tokenizers>=0.21' matplotlib
# jax를 커널에서 import하면 커널 프로세스가 TPU를 점유해 이후 !python 이 전부
# 'ABORTED: The TPU is already in use' 로 죽는다. 확인도 서브프로세스로 한다.
!python -c "import jax; print(jax.__version__, jax.devices()); assert jax.devices()[0].platform=='tpu', 'TPU 런타임이 아님'"


In [ ]:
import os, glob, shutil, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/rdepth-tpu'   # bin 원본 + logs/results (작은 파일만)
LOCAL = '/content/data'                        # 학습이 읽는 곳. Drive memmap 랜덤 접근은 금물
CKPT  = '/content/ckpt'                        # 체크포인트도 로컬. Drive FUSE에 큰 파일을
                                               # 반복해 쓰면 캐시가 로컬 디스크를 채운다
TPU   = '/content/tpu'
for d in (DRIVE, LOCAL, CKPT):
    os.makedirs(d, exist_ok=True)

# tpu.zip을 파일 패널 /content 루트에 드래그하면 Drive에 보관되어 다음 세션부터 자동.
if os.path.exists('/content/tpu.zip'):   # 새로 드래그한 zip이 항상 Drive를 갱신한다
    shutil.copy('/content/tpu.zip', f'{DRIVE}/tpu.zip')
if not os.path.exists(f'{DRIVE}/tpu.zip'):
    raise FileNotFoundError(
        'tpu.zip이 없습니다. 파일 패널 /content 루트에 드래그한 뒤 이 셀을 다시 실행하세요.\n'
        '드래그가 안 되면: from google.colab import files; files.upload()')
os.makedirs(TPU, exist_ok=True)
zipfile.ZipFile(f'{DRIVE}/tpu.zip').extractall(TPU)
os.chdir(TPU)
print(os.getcwd(), sorted(os.listdir('.')))
!df -h / | tail -1


In [ ]:
!python test_model.py

## 데이터

둘 다 Drive에 한 번만 만들고, 세션마다 로컬 디스크로 복사해서 쓴다.
FineWeb train.bin은 약 3.8 GB — Drive 여유를 먼저 확인할 것.

In [ ]:
def stage(name, build):
    """Drive에 없으면 build()로 만들고, 로컬로 복사한 경로를 돌려준다."""
    src, dst = f'{DRIVE}/{name}', f'{LOCAL}/{name}'
    if not os.path.exists(f'{src}/train.bin'):
        os.makedirs(src, exist_ok=True); build(src)
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f'{src}/*.bin'):
        if not os.path.exists(f'{dst}/{os.path.basename(f)}'):
            print('copy', f); shutil.copy(f, dst)
    return dst

def build_tinystories(out):
    # zip 모드면 같은 폴더, clone 모드면 ../rdepth 에 있다
    src = 'prepare_data.py' if os.path.exists('prepare_data.py') else '../rdepth/prepare_data.py'
    subprocess.run(['python', src], check=True)
    for f in glob.glob(os.path.join(os.path.dirname(os.path.abspath(src)), 'data', '*.bin')):
        shutil.move(f, out)

TINY = stage('tinystories', build_tinystories)
print(TINY, os.listdir(TINY))


In [ ]:
# 30분 이상 걸린다. 150m arm이 1.864B 토큰을 요구하므로 1.9B를 만든다.
FINEWEB = stage('fineweb', lambda out: subprocess.run(
    ['python', 'prepare_fineweb.py', '--out', out, '--target-tokens', '1900000000'], check=True))
print(FINEWEB, os.listdir(FINEWEB))

## CU 소모율 측정 (10분)

실행 **전에** 우상단 CU 잔량을 적어두고, 셀이 끝나면 다시 확인한다.
출력의 tok/s로 Phase 1 전체 시간을 다시 계산한다.

In [ ]:
import time
print('시작 시각', time.strftime('%H:%M:%S'), '— 지금 CU 잔량을 적어둘 것')
# TinyStories bin으로 측정한다. FineWeb 준비(30분+)를 기다릴 이유가 없다.
# --tokens로 요구량을 낮춰야 '데이터 부족' assert를 안 밟는다.
!python train.py --run 150m-loop --data-dir {TINY} --tokens 100000000 --out-dir /content/probe --max-seconds 600
print('종료 시각', time.strftime('%H:%M:%S'), '— CU 잔량 다시 확인')
# 150m-loop이 사다리에서 가장 무거운 arm이라 여기 tok/s가 최악값이다.
# Phase 1 전체 = 8.54e18 FLOPs. 150m-loop의 FLOPs/token = 6 * 12.25*1664^2 * 8 = 1.63e9
# → 실측 tok/s * 1.63e9 = 달성 FLOP/s. 8.54e18 을 그 값으로 나누면 전체 초.

## 15m 게이트 — TinyStories

PyTorch 참조 `1.5950 / 1.5577 / 1.5038` → recovery **0.4090**.
0.41±0.04를 벗어나면 멈추고 원인을 찾는다. Phase 2로 넘어가지 않는다.

In [ ]:
for arm in ['small', 'loop', 'large']:
    !python train.py --run 15m-{arm} --data-dir {TINY} --out-dir {DRIVE} --ckpt-dir {CKPT} --ckpt-every 200 --resume


In [ ]:
!python scaling.py --out-dir {DRIVE}


## 사다리 — FineWeb-Edu

게이트를 통과했을 때만. 150m이 전체 compute의 89%를 먹으므로, CU 측정치를
보고 15m+50m만 돌릴지 결정한다 (그것만으로도 추세선의 두 점은 나온다).

체크포인트는 `/content/ckpt` (로컬 225 GB). 9개 arm 전부 남겨도 11 GB라
지울 필요가 없다. Drive에는 작은 logs/results만 쓴다 — FUSE 캐시가
로컬 디스크를 채우는 걸 피하려는 것.


In [ ]:
CKPT_EVERY = {'15m': 200, '50m': 500, '150m': 4000}   # 150m 체크포인트 3.3 GB
ACCUM = {'15m': 1, '50m': 1, '150m': 1}               # HBM OOM 나면 150m을 2로

# 이미 끝난 스케일은 리스트에서 빼도 된다 (재실행해도 결과는 안 깨지지만 시간이 든다)
for scale in ['15m', '50m', '150m']:
    for arm in ['small', 'loop', 'large']:
        !python train.py --run {scale}-{arm} --data-dir {FINEWEB} --out-dir {DRIVE} --ckpt-dir {CKPT} --ckpt-every {CKPT_EVERY[scale]} --accum {ACCUM[scale]} --resume


## 시드 반복 — 오차막대

recovery는 비율이라 노이즈가 증폭된다. loop arm의 val loss가 0.005 움직이면
recovery는 0.055 움직인다. 점 하나에 시드 하나면 추세선의 기울기를 노이즈와
구분할 수 없다.

15m부터 채운다. 2.4 CU로 '150m 런이 해석 가능한가'를 먼저 산다.
산포가 ±0.03 이내면 50m 시드는 건너뛰고 150m으로 간다.


In [ ]:
for seed in [1338, 1339]:
    for arm in ['small', 'loop', 'large']:
        !python train.py --run 15m-{arm} --data-dir {FINEWEB} --out-dir {DRIVE} --ckpt-dir {CKPT} --seed {seed} --ckpt-every 200 --resume


In [ ]:
# 15m 산포가 ±0.05 이상일 때만. 14.6 CU.
for seed in [1338, 1339]:
    for arm in ['small', 'loop', 'large']:
        !python train.py --run 50m-{arm} --data-dir {FINEWEB} --out-dir {DRIVE} --ckpt-dir {CKPT} --seed {seed} --ckpt-every 500 --resume


In [ ]:
!python scaling.py --out-dir {DRIVE} --plot {DRIVE}/results/recovery.png
from IPython.display import Image
Image(f'{DRIVE}/results/recovery.png')
